# TrOCR Fine-Tuning Pipeline — FINAL VERSION
### Includes Table Transformer for table cell extraction

In [ ]:
# only run if you dont have ground_truth folder created

import fitz  # PyMuPDF
import pytesseract
import os
import cv2
from tqdm import tqdm

# 1. SETUP
pytesseract.pytesseract.tesseract_cmd = r"/opt/homebrew/bin/tesseract"
root_dir = '/Users/hanamengistu/Documents/Practicum1/practicum1.0/FinalProjectCode/Doc_label-4'
splits = ['train', 'valid']

for split in splits:
    images_path = os.path.join(root_dir, split, 'images')
    # CREATE THE FOLDER IF IT DOESN'T EXIST
    gt_path = os.path.join(root_dir, split, 'ground_truth')
    os.makedirs(gt_path, exist_ok=True)

    image_files = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.png'))]

    print(f"📝 Creating Draft Ground Truths for: {split}")
    for img_file in tqdm(image_files):
        img_path = os.path.join(images_path, img_file)

        # Simple OCR to get the text
        image = cv2.imread(img_path)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        text = pytesseract.image_to_string(gray)

        # Save as a .txt file with the SAME NAME as the image/label
        txt_name = os.path.splitext(img_file)[0] + ".txt"
        with open(os.path.join(gt_path, txt_name), "w") as f:
            f.write(text.strip())

print("\n✅ Drafts created! Now, open the 'ground_truth' folders and manually fix the text to make it 100% perfect.")

---
## CELL 0 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## CELL 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q "transformers[torch]" "accelerate>=1.1.0" \
    datasets evaluate jiwer tqdm \
    opencv-python pillow pandas pytesseract timm

# Install tesseract OCR engine on Colab
!apt-get install -q -y tesseract-ocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.6 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


---
## CELL 2 — Crop + Label-Aware OCR with Table Transformer
> - **Headers / Paragraphs / Table Info** → cropped and read with Tesseract
> - **Tables** → Table Transformer detects every row and column, each cell read individually
> - **diagram_image** → skipped (no text)
> - Works for ANY table size: 2x2, 10x3, 20x5, etc.

In [ ]:
import os
import cv2
import pytesseract
import pandas as pd
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
from transformers import (
    TableTransformerForObjectDetection,
    AutoImageProcessor
)

pytesseract.pytesseract.tesseract_cmd = r"/usr/bin/tesseract"

ROOT_DIR   = '/content/drive/MyDrive/final_project/Doc_label-4'
OUTPUT_DIR = './OCR_Training_Data'
SPLITS     = ['train', 'valid']

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

# Class map from data.yaml
CLASS_NAMES = {
    0: 'header',
    1: 'diagram_image',   # NO TEXT — skipped
    2: 'paragraph',
    3: 'table',
    4: 'table_header_rows',
    5: 'table_header_columns',
    6: 'table_data',
    7: 'table_info',
}

# Table classes — go through Table Transformer
TABLE_CLASSES = {3, 4, 5, 6}

# Skip entirely — no text
SKIP_CLASSES = {1}

# Tesseract PSM per non-table class
CLASS_PSM = {
    0: 6,   # header      → single text block
    2: 6,   # paragraph   → single text block
    7: 6,   # table_info  → single text block
}

# Load Table Transformer
print('Loading Table Transformer...')
TABLE_STRUCTURE_MODEL = 'microsoft/table-transformer-structure-recognition-v1.1-all'
tt_processor = AutoImageProcessor.from_pretrained(
    TABLE_STRUCTURE_MODEL,
    use_fast=False,
    size={'shortest_edge': 800, 'longest_edge': 1333}
)
tt_model = TableTransformerForObjectDetection.from_pretrained(
    TABLE_STRUCTURE_MODEL
).to(DEVICE)
tt_model.eval()
TT_LABELS = tt_model.config.id2label
print('Table Transformer loaded')


# ── HELPER: YOLO polygon/bbox → pixel coords ─────────────────────────────────
def get_bbox_pixels(parts, w_img, h_img, pad=5):
    coords = parts[1:]
    if len(coords) == 4:
        x_c, y_c, w_n, h_n = coords
        x1 = int((x_c - w_n / 2) * w_img)
        y1 = int((y_c - h_n / 2) * h_img)
        x2 = int((x_c + w_n / 2) * w_img)
        y2 = int((y_c + h_n / 2) * h_img)
    else:
        xs = coords[0::2]
        ys = coords[1::2]
        x1 = int(min(xs) * w_img)
        y1 = int(min(ys) * h_img)
        x2 = int(max(xs) * w_img)
        y2 = int(max(ys) * h_img)
    x1 = max(0,     x1 - pad)
    y1 = max(0,     y1 - pad)
    x2 = min(w_img, x2 + pad)
    y2 = min(h_img, y2 + pad)
    return x1, y1, x2, y2


# ── HELPER: detect cells in a table crop ─────────────────────────────────────
def detect_table_cells(crop_bgr, conf_threshold=0.5):
    """
    Runs Table Transformer on a table crop.
    Detects rows and columns, intersects them to get individual cells.
    Works for any table size — 2x2, 10x3, 20x5, etc.
    Returns list of cell dicts: {row, col, x1, y1, x2, y2}
    """
    h, w = crop_bgr.shape[:2]
    pil_img = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))

    inputs = tt_processor(images=pil_img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = tt_model(**inputs)

    target_sizes = torch.tensor([[h, w]]).to(DEVICE)
    results = tt_processor.post_process_object_detection(
        outputs,
        threshold=conf_threshold,
        target_sizes=target_sizes
    )[0]

    rows = []
    cols = []

    for score, label_id, box in zip(
        results['scores'], results['labels'], results['boxes']
    ):
        label_name = TT_LABELS[label_id.item()].lower()
        bx1, by1, bx2, by2 = [int(v) for v in box.tolist()]
        bx1 = max(0, bx1); by1 = max(0, by1)
        bx2 = min(w, bx2); by2 = min(h, by2)

        if 'row' in label_name:
            rows.append((by1, by2))
        elif 'column' in label_name:
            cols.append((bx1, bx2))

    if not rows or not cols:
        return []

    rows = sorted(rows, key=lambda r: r[0])
    cols = sorted(cols, key=lambda c: c[0])

    cells = []
    for row_idx, (ry1, ry2) in enumerate(rows):
        for col_idx, (cx1, cx2) in enumerate(cols):
            cells.append({
                'row': row_idx, 'col': col_idx,
                'x1': cx1, 'y1': ry1,
                'x2': cx2, 'y2': ry2,
            })
    return cells


# ── HELPER: read one cell with Tesseract ─────────────────────────────────────
def read_cell(crop_bgr, cell):
    cell_img = crop_bgr[cell['y1']:cell['y2'], cell['x1']:cell['x2']]
    if cell_img.size == 0:
        return ''
    gray = cv2.cvtColor(cell_img, cv2.COLOR_BGR2GRAY)
    # PSM 7 = single line — perfect for individual table cells
    return pytesseract.image_to_string(
        gray, config='--psm 7 --oem 3'
    ).strip()


# ── MAIN ─────────────────────────────────────────────────────────────────────
def prepare_training_crops():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    total = 0; skipped_diagrams = 0; empty_crops = 0; tt_failures = 0
    class_counts = {name: 0 for name in CLASS_NAMES.values()}
    class_counts['table_cell'] = 0

    for split in SPLITS:
        print(f'\nProcessing {split} split...')
        images_path   = os.path.join(ROOT_DIR, split, 'images')
        labels_path   = os.path.join(ROOT_DIR, split, 'labels')
        split_img_out = os.path.join(OUTPUT_DIR, split)
        os.makedirs(split_img_out, exist_ok=True)

        label_files = [f for f in os.listdir(labels_path) if f.endswith('.txt')]
        split_meta  = []

        for label_file in tqdm(label_files):
            img_name = label_file.replace('.txt', '.jpg')
            img_path = os.path.join(images_path, img_name)
            if not os.path.exists(img_path):
                img_path = img_path.replace('.jpg', '.png')
            image = cv2.imread(img_path)
            if image is None:
                print(f'  Could not load: {img_path}')
                continue
            h_img, w_img, _ = image.shape

            with open(os.path.join(labels_path, label_file), 'r') as f:
                yolo_lines = [line.strip() for line in f if line.strip()]

            for box_idx, yolo_line in enumerate(yolo_lines):
                parts      = list(map(float, yolo_line.split()))
                class_id   = int(parts[0])
                label_name = CLASS_NAMES.get(class_id, f'unknown_{class_id}')
                base_name  = label_file.replace('.txt', '')

                if class_id in SKIP_CLASSES:
                    skipped_diagrams += 1
                    continue

                x1, y1, x2, y2 = get_bbox_pixels(parts, w_img, h_img)
                crop = image[y1:y2, x1:x2]
                if crop.size == 0:
                    continue

                # ── TABLE → Table Transformer ─────────────────────────────
                if class_id in TABLE_CLASSES:
                    cells = detect_table_cells(crop)

                    if not cells:
                        # Fallback: read whole crop if TT finds nothing
                        tt_failures += 1
                        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
                        text = pytesseract.image_to_string(
                            gray, config='--psm 6 --oem 3'
                        ).strip()
                        if text and 3 <= len(text) <= 300:
                            fname = f"{label_name}_{base_name}_{box_idx}_fallback.png"
                            cv2.imwrite(os.path.join(split_img_out, fname), crop)
                            split_meta.append({
                                'file_name': fname, 'text': text,
                                'label': label_name
                            })
                            total += 1
                        continue

                    for cell in cells:
                        text = read_cell(crop, cell)
                        if not text or not (3 <= len(text) <= 300):
                            empty_crops += 1
                            continue
                        fname = f"table_cell_{base_name}_b{box_idx}_r{cell['row']}_c{cell['col']}.png"
                        cell_img = crop[cell['y1']:cell['y2'], cell['x1']:cell['x2']]
                        cv2.imwrite(os.path.join(split_img_out, fname), cell_img)
                        split_meta.append({
                            'file_name': fname, 'text': text,
                            'label': 'table_cell',
                            'table_row': cell['row'],
                            'table_col': cell['col'],
                        })
                        class_counts['table_cell'] += 1
                        total += 1

                # ── NON-TABLE → Tesseract directly ────────────────────────
                else:
                    psm  = CLASS_PSM.get(class_id, 6)
                    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
                    text = pytesseract.image_to_string(
                        gray, config=f'--psm {psm} --oem 3'
                    ).strip()
                    if not text or not (3 <= len(text) <= 300):
                        empty_crops += 1
                        continue
                    fname = f"{label_name}_{base_name}_{box_idx}.png"
                    cv2.imwrite(os.path.join(split_img_out, fname), crop)
                    split_meta.append({
                        'file_name': fname, 'text': text, 'label': label_name
                    })
                    class_counts[label_name] += 1
                    total += 1

        pd.DataFrame(split_meta).to_csv(
            os.path.join(split_img_out, 'metadata.csv'), index=False
        )
        print(f'  {len(split_meta)} examples saved for {split}')

    print(f'\nTotal examples: {total}')
    print(f'Diagrams skipped: {skipped_diagrams}')
    print(f'Empty/oversized skipped: {empty_crops}')
    print(f'TT fallbacks: {tt_failures}')
    print('\nExamples per label:')
    for name, count in sorted(class_counts.items(), key=lambda x: -x[1]):
        bar = '█' * min(count, 50)
        print(f'  {name:<25} {count:>5}  {bar}')

prepare_training_crops()

Using device: cuda
Loading Table Transformer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

...

Table Transformer loaded

Processing train split...


100%|██████████| 77/77 [35:25<00:00, 27.60s/it]


  10021 examples saved for train

Processing valid split...


100%|██████████| 20/20 [12:24<00:00, 37.24s/it]

  3843 examples saved for valid

Total examples: 13864
Diagrams skipped: 3
Empty/oversized skipped: 9471
TT fallbacks: 17

Examples per label:
  table_cell                13583  ██████████████████████████████████████████████████
  header                      217  ██████████████████████████████████████████████████
  paragraph                    32  ████████████████████████████████
  table_info                   21  █████████████████████
  diagram_image                 0  
  table                         0  
  table_header_rows             0  
  table_header_columns          0  
  table_data                    0  


---
## CELL 3 — Clean and Filter Metadata
> Run this after Cell 2. Removes oversized crops and any empty/NaN text rows.
> **Must run before loading dataset.**

In [ ]:
import pandas as pd
import os

# Keep only label types TrOCR can handle well
KEEP_LABELS = {'header', 'paragraph', 'table_info', 'table_cell'}
MAX_CHARS   = 300

for split in ['train', 'valid']:
    meta_path = f'./OCR_Training_Data/{split}/metadata.csv'
    if not os.path.exists(meta_path):
        print(f'No metadata for {split} — run Cell 2 first')
        continue

    df = pd.read_csv(meta_path)
    before = len(df)

    # Step 1: keep only trainable label types and short text
    df = df[df['label'].isin(KEEP_LABELS)].copy()
    df = df[df['text'].astype(str).str.len() <= MAX_CHARS].copy()

    # Step 2: drop missing, NaN, or empty text (prevents tokenizer crash)
    df = df.dropna(subset=['text'])
    df = df[df['text'].astype(str).str.strip() != '']
    df = df[df['text'].astype(str) != 'nan']

    # Step 3: drop very short text (single chars = Tesseract misread)
    df = df[df['text'].astype(str).str.len() >= 3]

    df.to_csv(meta_path, index=False)
    print(f'\n{split}: kept {len(df)}/{before} examples')
    print(df['label'].value_counts().to_string())

print('\n✅ Done — now run Cell 4 (load dataset)')


train: kept 10013/10021 examples
label
table_cell    9819
header         156
paragraph       29
table_info       9

valid: kept 3834/3843 examples
label
table_cell    3758
header          61
table_info      12
paragraph        3

✅ Done — now run Cell 4 (load dataset)


---
## CELL 4 — Load Dataset

In [ ]:
from datasets import load_dataset

OUTPUT_DIR = './OCR_Training_Data'

dataset = load_dataset('imagefolder', data_dir=OUTPUT_DIR)
print(dataset)

# Rename 'valid' to 'validation' if needed
if 'valid' in dataset and 'validation' not in dataset:
    dataset['validation'] = dataset.pop('valid')
    print('Renamed: valid → validation')

print('\nFinal splits:', list(dataset.keys()))

Resolving data files:   0%|          | 0/10022 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3844 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'text', 'label', 'table_row', 'table_col'],
        num_rows: 10013
    })
    validation: Dataset({
        features: ['image', 'text', 'label', 'table_row', 'table_col'],
        num_rows: 3834
    })
})

Final splits: ['train', 'validation']


---
## CELL 5 — Load TrOCR Processor and Model

In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Using device: {DEVICE}')

BASE_MODEL = 'microsoft/trocr-base-printed'
processor  = TrOCRProcessor.from_pretrained(BASE_MODEL)
model      = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL).to(DEVICE)

# Required for generation to work correctly
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

print('✅ Model and processor loaded')

Using device: cuda


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model and processor loaded


---
## CELL 6 — Preprocessing

In [ ]:
def preprocess(batch):
    # FIX: ensure all text values are clean strings — None/NaN crashes tokenizer
    texts = [str(t) if t is not None and str(t) != 'nan' else '' for t in batch['text']]

    # Only process examples with valid text
    valid_indices = [i for i, t in enumerate(texts) if len(t.strip()) >= 3]
    images = [batch['image'][i].convert('RGB') for i in valid_indices]
    texts  = [texts[i] for i in valid_indices]

    pixel_values = processor(
        images=images,
        return_tensors='pt'
    ).pixel_values

    labels = processor.tokenizer(
        texts,
        padding='max_length',
        max_length=128,
        truncation=True
    ).input_ids

    # Replace padding token with -100 so loss ignores it
    labels = [
        [(tok if tok != processor.tokenizer.pad_token_id else -100) for tok in label]
        for label in labels
    ]

    batch['pixel_values'] = pixel_values
    batch['labels']       = labels
    return batch

dataset = dataset.map(preprocess, batched=True, batch_size=16)
dataset.set_format(type='torch', columns=['pixel_values', 'labels'])
print('✅ Preprocessing complete')

Map:   0%|          | 0/10013 [00:00<?, ? examples/s]

Map:   0%|          | 0/3834 [00:00<?, ? examples/s]

✅ Preprocessing complete


---
## CELL 7 — CER Accuracy Metric
> CER = Character Error Rate. 0.05 = 5% of characters wrong. Lower is better.

In [ ]:
import evaluate
import numpy as np

cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Clip to valid vocab range — prevents OverflowError
    vocab_size = processor.tokenizer.vocab_size
    pred_ids   = np.clip(pred_ids, 0, vocab_size - 1)

    # Replace -100 padding back to pad_token_id before decoding
    label_ids = label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    print('\n── Sample Predictions ──────────────────────')
    for p, l in zip(pred_str[:3], label_str[:3]):
        print(f'  PRED : {p[:80]}')
        print(f'  TRUE : {l[:80]}')
        print()

    return {'cer': round(cer, 4)}

print('✅ CER metric ready')

✅ CER metric ready


---
## CELL 8 — Training

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, default_data_collator

training_args = Seq2SeqTrainingArguments(
    output_dir                  = './trocr_checkpoints',
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    num_train_epochs            = 5,
    eval_strategy               = 'steps',
    save_strategy               = 'steps',
    save_steps                  = 500,
    eval_steps                  = 500,
    logging_steps               = 10,
    learning_rate               = 4e-5,
    save_total_limit            = 2,
    remove_unused_columns       = False,
    push_to_hub                 = False,
    predict_with_generate       = True,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'cer',
    greater_is_better           = False,
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = dataset['train'],
    eval_dataset     = dataset['validation'],
    data_collator    = default_data_collator,
    compute_metrics  = compute_metrics,
    processing_class = processor,
)

trainer.train()
print('\n✅ Training complete!')

Step,Training Loss,Validation Loss,Cer
500,1.587720,2.453313,0.512400
1000,1.612726,2.277330,0.489700
1500,1.220168,2.193545,0.462100
2000,0.597257,2.096928,0.436600
2500,0.649990,1.984662,0.424400
3000,0.724421,1.957056,0.408400



── Sample Predictions ──────────────────────
  PRED : Bankruptcy Events

ing
ing
ing
ing
ing
ing
ing
  TRUE : Bankruptcy Events
Alphabetical Listing
Revised 5/17/2016

  PRED : Notice of Proposed Abandon Payment of Property of Not Notute Notute Notices Noti
  TRUE : Notice of Proposed Abandonment of Property of the Estate Notices BK

  PRED : Notice of Proposed Abandon of Property of Estate
  TRUE : Notice of Proposed Abandonment of Property of the Estate



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


── Sample Predictions ──────────────────────
  PRED : Bankruptcy Events
Alphabetical Listing
Alphabetical Listing
ical List
  TRUE : Bankruptcy Events
Alphabetical Listing
Revised 5/17/2016

  PRED : Notice of Proposed Abandonment of Property of the Estate BK
  TRUE : Notice of Proposed Abandonment of Property of the Estate Notices BK

  PRED : Notice of Proposed Abandon of Property of the Estate
  TRUE : Notice of Proposed Abandonment of Property of the Estate



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


── Sample Predictions ──────────────────────
  PRED : Bankruptcy Events
Alphabetical Listing
Revised 5/17/2016
  TRUE : Bankruptcy Events
Alphabetical Listing
Revised 5/17/2016

  PRED : Notice of Proposed Abandonment of Property of the Estate Notices BK
  TRUE : Notice of Proposed Abandonment of Property of the Estate Notices BK

  PRED : Notice of Proposed Abandonment of Property of the Estate
  TRUE : Notice of Proposed Abandonment of Property of the Estate



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].



✅ Training complete!


---
## CELL 9 — Final Validation Results

In [ ]:
results = trainer.evaluate()

cer_val  = results.get('eval_cer',  None)
loss_val = results.get('eval_loss', None)

print('\n═══════════════════════════════════════════')
print('  FINAL VALIDATION RESULTS')
print('═══════════════════════════════════════════')
print(f'  Loss : {loss_val:.4f}')
print(f'  CER  : {cer_val:.4f}  ({cer_val*100:.1f}% character error rate)')
if cer_val < 0.05:
    print('  🟢 Excellent accuracy for printed documents')
elif cer_val < 0.15:
    print('  🟡 Acceptable — consider more epochs or more data')
else:
    print('  🔴 High error rate — review ground truth quality or increase training')
print('═══════════════════════════════════════════')


── Sample Predictions ──────────────────────
  PRED : Bankruptcy Events
Alphabetical Listing
Revised 5/17/2016
  TRUE : Bankruptcy Events
Alphabetical Listing
Revised 5/17/2016

  PRED : Notice of Proposed Abandonment of Property of the Estate Notices BK
  TRUE : Notice of Proposed Abandonment of Property of the Estate Notices BK

  PRED : Notice of Proposed Abandonment of Property of the Estate
  TRUE : Notice of Proposed Abandonment of Property of the Estate


═══════════════════════════════════════════
  FINAL VALIDATION RESULTS
═══════════════════════════════════════════
  Loss : 1.9571
  CER  : 0.4084  (40.8% character error rate)
  🔴 High error rate — review ground truth quality or increase training
═══════════════════════════════════════════


---
## CELL 10 — Save Model

In [ ]:
model.save_pretrained('./my_finetuned_ocr')
processor.save_pretrained('./my_finetuned_ocr')
print('✅ Model saved to ./my_finetuned_ocr')

# Also save to Google Drive so it persists after Colab session ends
import shutil
drive_save_path = '/content/drive/MyDrive/final_project/my_finetuned_ocr'
if os.path.exists(drive_save_path):
    shutil.rmtree(drive_save_path)
shutil.copytree('./my_finetuned_ocr', drive_save_path)
print(f'✅ Model also saved to Google Drive: {drive_save_path}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ./my_finetuned_ocr
✅ Model also saved to Google Drive: /content/drive/MyDrive/final_project/my_finetuned_ocr


---
## CELL 11 — Inference on a Test Image

In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import cv2
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_PATH      = './my_finetuned_ocr'
TEST_IMAGE_PATH = '/content/drive/MyDrive/final_project/Doc_label-4/test/images/0a29925ccc5e6299e132a73325956a3abef6dd26_page_2_jpg.rf.ed2c6c25fad2cd688495425d80ba2bef.jpg'
TEST_LABEL_PATH = '/content/drive/MyDrive/final_project/Doc_label-4/test/labels/0a29925ccc5e6299e132a73325956a3abef6dd26_page_2_jpg.rf.ed2c6c25fad2cd688495425d80ba2bef.txt'

print(f'Loading model from {MODEL_PATH}...')
inf_processor = TrOCRProcessor.from_pretrained(MODEL_PATH)
inf_model     = VisionEncoderDecoderModel.from_pretrained(MODEL_PATH).to(DEVICE)
inf_model.eval()

def predict_text(pil_image):
    pixel_values = inf_processor(
        images=pil_image.convert('RGB'), return_tensors='pt'
    ).pixel_values.to(DEVICE)
    with torch.no_grad():
        generated_ids = inf_model.generate(pixel_values)
    return inf_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Load test image
image = cv2.imread(TEST_IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f'Could not load: {TEST_IMAGE_PATH}')
h_img, w_img, _ = image.shape

# Load YOLO boxes
with open(TEST_LABEL_PATH, 'r') as f:
    lines = [l.strip() for l in f if l.strip()]

# Sort boxes top-to-bottom (polygon-safe)
def get_center_y(line):
    parts  = list(map(float, line.split()))
    coords = parts[1:]
    ys     = coords[1::2] if len(coords) > 4 else [coords[1]]
    return sum(ys) / len(ys)
lines.sort(key=get_center_y)

print(f'\n📄 OCR Results for {os.path.basename(TEST_IMAGE_PATH)}:')
print('-' * 50)

full_text = []
for line in lines:
    parts    = list(map(float, line.split()))
    class_id = int(parts[0])

    # Skip diagram boxes in inference too
    if class_id == 1:
        continue

    coords = parts[1:]
    if len(coords) == 4:
        x_c, y_c, w_n, h_n = coords
        x1 = int((x_c - w_n / 2) * w_img)
        y1 = int((y_c - h_n / 2) * h_img)
        x2 = int((x_c + w_n / 2) * w_img)
        y2 = int((y_c + h_n / 2) * h_img)
    else:
        xs = coords[0::2]; ys = coords[1::2]
        x1 = int(min(xs) * w_img); y1 = int(min(ys) * h_img)
        x2 = int(max(xs) * w_img); y2 = int(max(ys) * h_img)

    pad = 5
    x1 = max(0, x1-pad); y1 = max(0, y1-pad)
    x2 = min(w_img, x2+pad); y2 = min(h_img, y2+pad)

    crop = image[y1:y2, x1:x2]
    if crop.size == 0:
        continue

    label_name = CLASS_NAMES.get(class_id, 'unknown')

    # For table boxes in inference — use Table Transformer + cell-by-cell prediction
    if class_id in {3, 4, 5, 6}:
        cells = detect_table_cells(crop)
        if cells:
            print(f'\n[TABLE — {len(cells)} cells detected]')
            # Group by row for readable output
            from collections import defaultdict
            rows = defaultdict(dict)
            for cell in cells:
                cell_img = crop[cell['y1']:cell['y2'], cell['x1']:cell['x2']]
                if cell_img.size == 0:
                    continue
                pil_cell = Image.fromarray(cv2.cvtColor(cell_img, cv2.COLOR_BGR2RGB))
                rows[cell['row']][cell['col']] = predict_text(pil_cell)
            for row_idx in sorted(rows.keys()):
                row_text = ' | '.join(rows[row_idx].get(c, '') for c in sorted(rows[row_idx]))
                print(f'  Row {row_idx}: {row_text}')
                full_text.append(row_text)
        else:
            # Fallback for tables TT couldn't parse
            pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            text = predict_text(pil_crop)
            print(f'  [{label_name}]: {text}')
            full_text.append(text)
    else:
        pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        text = predict_text(pil_crop)
        print(f'  [{label_name}]: {text}')
        full_text.append(text)

print('\n── FINAL RECONSTRUCTED TEXT ──')
print('\n'.join(full_text))

Loading model from ./my_finetuned_ocr...


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]


📄 OCR Results for 0a29925ccc5e6299e132a73325956a3abef6dd26_page_2_jpg.rf.ed2c6c25fad2cd688495425d80ba2bef.jpg:
--------------------------------------------------
  [paragraph]: carbonitarian at the authorities of engage the
e determined of the beneficiary by how actions
e
  [paragraph]: Contract the OCHA Centre for Humanitarian Data is LED, Item, Item Lean

── FINAL RECONSTRUCTED TEXT ──
carbonitarian at the authorities of engage the
e determined of the beneficiary by how actions
e
Contract the OCHA Centre for Humanitarian Data is LED, Item, Item Lean


In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import cv2
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_PATH      = './my_finetuned_ocr'
TEST_IMAGE_PATH = '/content/drive/MyDrive/final_project/Doc_label-4/test/images/2UD7BSKC7GT3V7I6DK5NH5TFKJ6F3EXL-pdf_page_3_png.rf.863e15b507000261f12642ec5c80f7f6.jpg'
TEST_LABEL_PATH = '/content/drive/MyDrive/final_project/Doc_label-4/test/labels/2UD7BSKC7GT3V7I6DK5NH5TFKJ6F3EXL-pdf_page_3_png.rf.863e15b507000261f12642ec5c80f7f6.txt'

print(f'Loading model from {MODEL_PATH}...')
inf_processor = TrOCRProcessor.from_pretrained(MODEL_PATH)
inf_model     = VisionEncoderDecoderModel.from_pretrained(MODEL_PATH).to(DEVICE)
inf_model.eval()

def predict_text(pil_image):
    pixel_values = inf_processor(
        images=pil_image.convert('RGB'), return_tensors='pt'
    ).pixel_values.to(DEVICE)
    with torch.no_grad():
        generated_ids = inf_model.generate(pixel_values)
    return inf_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Load test image
image = cv2.imread(TEST_IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f'Could not load: {TEST_IMAGE_PATH}')
h_img, w_img, _ = image.shape

# Load YOLO boxes
with open(TEST_LABEL_PATH, 'r') as f:
    lines = [l.strip() for l in f if l.strip()]

# Sort boxes top-to-bottom (polygon-safe)
def get_center_y(line):
    parts  = list(map(float, line.split()))
    coords = parts[1:]
    ys     = coords[1::2] if len(coords) > 4 else [coords[1]]
    return sum(ys) / len(ys)
lines.sort(key=get_center_y)

print(f'\n📄 OCR Results for {os.path.basename(TEST_IMAGE_PATH)}:')
print('-' * 50)

full_text = []
for line in lines:
    parts    = list(map(float, line.split()))
    class_id = int(parts[0])

    # Skip diagram boxes in inference too
    if class_id == 1:
        continue

    coords = parts[1:]
    if len(coords) == 4:
        x_c, y_c, w_n, h_n = coords
        x1 = int((x_c - w_n / 2) * w_img)
        y1 = int((y_c - h_n / 2) * h_img)
        x2 = int((x_c + w_n / 2) * w_img)
        y2 = int((y_c + h_n / 2) * h_img)
    else:
        xs = coords[0::2]; ys = coords[1::2]
        x1 = int(min(xs) * w_img); y1 = int(min(ys) * h_img)
        x2 = int(max(xs) * w_img); y2 = int(max(ys) * h_img)

    pad = 5
    x1 = max(0, x1-pad); y1 = max(0, y1-pad)
    x2 = min(w_img, x2+pad); y2 = min(h_img, y2+pad)

    crop = image[y1:y2, x1:x2]
    if crop.size == 0:
        continue

    label_name = CLASS_NAMES.get(class_id, 'unknown')

    # For table boxes in inference — use Table Transformer + cell-by-cell prediction
    if class_id in {3, 4, 5, 6}:
        cells = detect_table_cells(crop)
        if cells:
            print(f'\n[TABLE — {len(cells)} cells detected]')
            # Group by row for readable output
            from collections import defaultdict
            rows = defaultdict(dict)
            for cell in cells:
                cell_img = crop[cell['y1']:cell['y2'], cell['x1']:cell['x2']]
                if cell_img.size == 0:
                    continue
                pil_cell = Image.fromarray(cv2.cvtColor(cell_img, cv2.COLOR_BGR2RGB))
                rows[cell['row']][cell['col']] = predict_text(pil_cell)
            for row_idx in sorted(rows.keys()):
                row_text = ' | '.join(rows[row_idx].get(c, '') for c in sorted(rows[row_idx]))
                print(f'  Row {row_idx}: {row_text}')
                full_text.append(row_text)
        else:
            # Fallback for tables TT couldn't parse
            pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            text = predict_text(pil_crop)
            print(f'  [{label_name}]: {text}')
            full_text.append(text)
    else:
        pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        text = predict_text(pil_crop)
        print(f'  [{label_name}]: {text}')
        full_text.append(text)

print('\n── FINAL RECONSTRUCTED TEXT ──')
print('\n'.join(full_text))

Loading model from ./my_finetuned_ocr...


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]


📄 OCR Results for 2UD7BSKC7GT3V7I6DK5NH5TFKJ6F3EXL-pdf_page_3_png.rf.863e15b507000261f12642ec5c80f7f6.jpg:
--------------------------------------------------
  [header]: Bankruptcy Events
Alphabetical Listing
Revised 5/17/2016

[TABLE — 2 cells detected]
  Row 0: Application to Current Socal Security Number(s) | Motions/Applications/Presentments

[TABLE — 60 cells detected]
  Row 0: Application to Current Socal Security Number(S) | Application to Current Secal Security Numberments) Motions/Applications/Presentments | Motions/Applications/Presentments
  Row 1: Application to Have the Chapter 7 Filing Fee Wised | Application to Time Chapter 7 Filing Fee Wested Motions/Applications/Presentments | Motions/Applications/Presentments
  Row 2: Appoint Creditor’s Cominite (Application) | Appoint Creditor’s Commine (Application) Motions/Applications/Present | Motions/Applications/Presentments
  Row 3: Appoint Examiner | Appoint Examor Motions/Applications/Presentments | Motions/Applications/Pre